# Yläkvantiilin mallintaminen: mittapoikkeaman häntä PROC QUANTREG:lla

## Tiivistelmä

Tarkkuuskoneistustehtaalle tärkeintä on **pahin mahdollinen** kappalekohtainen mittapoikkeama, ei pelkkä keskiarvo, koska yläkvantiili ohjaa hylkyä ja asiakasreklamaatioita. Tämä muistikirja käyttää **PROC QUANTREG** -menetelmää sovittamaan kvantiiliregression mediaaniin ja 90. persentiiliin, mikä paljastaa, että koneen kuluminen, kiertonopeus ja syöttönopeus vaikuttavat huomattavasti voimakkaammin yläkvantiiliin (hylkyriski) kuin mediaaniin — tämä on merkki heteroskedastisesta prosessista, jossa vaihtelu kasvaa työkalun kuluessa.

## Tietolähteet

| Aineisto | Rivit | Kuvaus | Avainmuuttujat |
|---------|------|-------------|---------------|
| `work.machining` | 100 | Synteettisiä kappalekohtaisia tietueita CNC-sorvauslinjalta, luotu suoraan komennoilla `call streaminit`/`rand`. Mittapoikkeama (mikronia) nimellisarvosta on mallinnettu heteroskedastisella virheellä, jonka hajonta kasvaa työkalun kulumisen ja kiertonopeuden myötä, joten prosessitekijät vaikuttavat voimakkaammin yläkvantiiliin kuin mediaaniin. | `Deviation` (vastemuuttuja, mikronia), `ToolWear` (kertynyt leikkausaika, min), `CycleSpeed` (rpm), `CoolantTemp` (°C), `Humidity` (%RH), `FeedRate` (mm/kierros), `Machine` (LUOKKA: M1–M4), `Shift` (LUOKKA: Day/Eve/Night), `PartID` |

# Yläkvantiilin mittapoikkeaman prosessitekijöiden mallintaminen

## Valmistuksen käyttötapaus: hylkyriskin mallinnus CNC-sorvauslinjalla

Tarkkuuskoneistuslinjalla kappale hylätään, kun **mittapoikkeama** nimellisarvosta kasvaa liian suureksi. Tehdas ei menetä rahaa *keskimääräisestä* kappaleesta — se menettää rahaa poikkeamajakauman **yläkvantiilista**. Tavallinen pienimmän neliösumman regressio mallintaa ehdollista keskiarvoa ja voi täysin ohittaa tekijät, jotka merkitsevät vain silloin, kun asiat menevät pieleen.

**Kvantiiliregressio** mallintaa valitun ehdollisen kvantiilin (esimerkiksi poikkeaman 90. persentiilin) keskiarvon sijaan. **PROC QUANTREG** sovittaa yhden tai useamman kvantiilin yhdellä kutsulla ja raportoi parametriestimaatin, keskivirheen ja luottamusrajat jokaiselle tekijälle jokaisessa kvantiilissa. Teemme seuraavaa:

1. Luomme realistisen synteettisen kappalekohtaisen aineiston, jonka virhehajonta kasvaa työkalun kulumisen ja kiertonopeuden myötä (jotta tekijät vaikuttavat voimakkaammin häntään kuin keskikohtaan).
2. Sovitamme **mediaanin** (q = 0,50) ja **hylkyriskihännän** (q = 0,90) yhdellä PROC QUANTREG -kutsulla.
3. Vertailemme kahta kertoimijoukkoa rinnakkain nähdäksemme, miten kulmakertoimet muuttuvat keskikohdasta häntään.
4. Pisteytämme jokaisen kappaleen sen sovitetulla 90. persentiilin ennusteella, jotta analyytikot voivat merkitä korkean hännän riskin kappaleet.

Kaikki alla oleva on itsenäistä — ei ulkoisia tiedostoja, ei verkkoyhteyttä.

## Vaihe 1 — Synteettisen koneistusaineiston luonti

Simuloimme sorvattuja kappaleita 4 koneella ja 3 vuorolla. Keskeinen realistisuustemppu on **heteroskedastisuus**: satunnaisen virhetermin keskihajonta (`sigma`) kasvaa muuttujien `ToolWear` ja `CycleSpeed` myötä. Tämän vuoksi nämä kaksi tekijää vaikuttavat huomattavasti voimakkaammin muuttujan `Deviation` 90. persentiiliin kuin sen mediaaniin — juuri tilanne, jossa kvantiiliregressio osoittaa arvonsa. Muuttujalla `Humidity` ei ole todellista signaalia data-generointiprosessissa, joten se toimii lumetekijänä, jota voimme tarkkailla.

In [1]:
TIEDOT work.machining;
    CALL streaminit(20260531);
    PITUUS Machine $2 Shift $5;
    TEE PartID = 1 ASTI 100;
        /* CLASS factors */
        m = rand('integer', 1, 4);
        Machine = cats('M', KIRJOITA(m, 1.));
        s = rand('integer', 1, 3);
        JOS s = 1 NIIN Shift = 'Day';
        MUUTEN JOS s = 2 NIIN Shift = 'Eve';
        MUUTEN Shift = 'Night';

        /* Continuous process drivers */
        ToolWear     = rand('uniform') * 120;            /* cumulative cut minutes */
        CycleSpeed   = 1200 + rand('normal') * 180;      /* spindle rpm */
        CoolantTemp  = 22 + rand('normal') * 3;          /* deg C */
        Humidity     = 45 + rand('normal') * 8;          /* %RH (placebo) */
        FeedRate     = 0.18 + rand('uniform') * 0.07;    /* mm/rev */

        /* Machine offsets: one machine runs hotter */
        machoff = (Machine = 'M3') * 2.0 + (Machine = 'M4') * 0.8;
        /* Night shift drifts slightly */
        shiftoff = (Shift = 'Night') * 1.2;

        /* Location (central tendency) of deviation in microns */
        MU = 5
           + 0.030 * ToolWear
           + 0.0015 * (CycleSpeed - 1200)
           + 0.45 * (CoolantTemp - 22)
           + 6.0 * FeedRate
           + machoff + shiftoff;

        /* Heteroscedastic spread: tail widens with wear & speed */
        SIGMA = 1.0
              + 0.020 * ToolWear
              + 0.0010 * abs(CycleSpeed - 1200);

        Deviation = MU + SIGMA * rand('normal');
        JOS Deviation < 0 NIIN Deviation = 0;

        SÄILYTÄ PartID Machine Shift ToolWear CycleSpeed CoolantTemp
             Humidity FeedRate Deviation;
        TULOSTE;
    LOPPU;
SUORITA;


NOTE: DATA work.machining


NOTE: Wrote work.machining (100 rows, 9 columns).
NOTE: DATA elapsed:
  wall  0.11 seconds
  cpu   0.11 seconds


### Pikakatsaus raakajakaumaan

Ennen mallinnusta varmistetaan, että vaste on oikealle vino ja sillä on merkittävä yläkvantiili — se on jakauman osa, joka ohjaa hylkyä. Tulostamme mediaanin ja yläpersentiilit suoraan PROC UNIVARIATE:sta, jotta näemme, kuinka paljon korkeampi 90. persentiili on mediaaniin verrattuna.

In [2]:
PROSEDUURI UNIVARIATE TIEDOT=work.machining NOPRINT;
    MUUTTUJA Deviation;
    TULOSTE out=work.devpct
        MEDIAN=Med p90=P90 p95=P95 p99=P99;
SUORITA;

PROSEDUURI TULOSTA TIEDOT=work.devpct noobs NIMIKE;
    NIMIKE Med = "Poikkeaman mediaani"
          P90 = "90. persentiili"
          P95 = "95. persentiili"
          P99 = "99. persentiili";
SUORITA;


Poikkeaman mediaani  90. persentiili  95. persentiili  99. persentiili
-------------------  ---------------  ---------------  ---------------
       8.7251490709    12.4132063767    13.5691645665    17.0510365805




NOTE: PROC UNIVARIATE
NOTE: Output dataset work.devpct has 1 observations and 4 variables.
NOTE: PROC PRINT data=work.devpct

NOTE: PROC PRINT completed: 1 observations printed, 4 variables


## Vaihe 2 — Mediaanin ja hylkyriskihännän sovitus yhdessä

PROC QUANTREG sovittaa molemmat kvantiilit yhdellä kutsulla: `QUANTILE=0.5 0.90`. `CLASS`-lause määrittelee kategoriset prosessitekijät (`Machine`, `Shift`); `MODEL`-lause listaa kaikki jatkuvat ehdokastekijät. Pyydämme `CI=SPARSITY`, joka käyttää sparsity-funktioestimaattoria tuottaakseen keskivirheen ja 95 %:n luottamuskaistan jokaiselle kertoimelle.

Lue kaksi tulosteryhmää ennen/jälkeen-asetelmana: ensimmäinen ryhmä (q = 0,50) kuvaa *tyypillistä* kappaletta; toinen (q = 0,90) kuvaa *hylkyalttiin* kappaleen. Seuraa muuttujia `ToolWear`, `CycleSpeed` ja `FeedRate` — koska simuloitu virhehajonta kasvaa kulumisen ja nopeuden myötä, niiden kulmakertoimien pitäisi olla suurempia yläkvantiilissa.

In [3]:
PROSEDUURI quantreg TIEDOT=work.machining ci=sparsity;
    LUOKKA Machine Shift;
    NIMIKE Deviation = "Poikkeama (mikronia)"
          Machine = "Kone"
          Shift = "Vuoro"
          ToolWear = "Työkalun kuluminen (min)"
          CycleSpeed = "Kiertonopeus (rpm)"
          CoolantTemp = "Jäähdytysnesteen lämpötila (aste C)"
          Humidity = "Kosteus (%RH)"
          FeedRate = "Syöttönopeus (mm/kierros)";
    MODEL Deviation = Machine Shift ToolWear CycleSpeed CoolantTemp
                      Humidity FeedRate
        / quantile=0.5 0.90;
SUORITA;


The QUANTREG Procedure

Quantile: 0.5000
CI Method: SPARSITY
Dependent Variable: Poikkeama (mikronia)

Parameter           Estimate       StdErr        Lower        Upper
Intercept            -2.4433       2.0123      -6.3874       1.5007
MACHINE M3            1.3258       0.3574       0.6254       2.0262
MACHINE M2           -0.1735       0.3245      -0.8095       0.4625
MACHINE M1           -0.5599       0.3173      -1.1818       0.0619
SHIFT EVE            -1.6360       0.2964      -2.2170      -1.0550
SHIFT DAY            -0.9975       0.2909      -1.5676      -0.4275
Työkalun kuluminen (min)       0.0240       0.0033       0.0174       0.0305
Kiertonopeus (rpm)      -0.0007       0.0008      -0.0022       0.0009
Jäähdytysnesteen lämpötila (aste C)       0.4542       0.0395       0.3767       0.5316
Kosteus (%RH)         0.0575       0.0150       0.0281       0.0868
Syöttönopeus (mm/kierros)      -5.0538       5.8578     -16.5350       6.4273
Intercept           -12.8502       0.7


NOTE: PROC QUANTREG data=work.machining

NOTE: PROC QUANTREG completed.


## Vaihe 3 — Keskikohta ja häntä rinnakkain

Kahden kertoimiryhmän lukeminen rinnakkain on hankalaa, joten talletamme parametritaulukon lauseella `ODS OUTPUT ParameterEstimates=` (joka sisältää sarakkeen `Quantile`), ja yhdistämme sitten q = 0,50 ja q = 0,90 -estimaatit jatkuville tekijöille yhdeksi vertailutaulukoksi. Sarake `Tail - Median` on pääluku: suuri positiivinen arvo tarkoittaa, että tekijä työntää hylkyriskihäntää paljon voimakkaammin kuin se siirtää tyypillistä kappaletta.

In [4]:
ODS TULOSTE ParameterEstimates=work.pe;
PROSEDUURI quantreg TIEDOT=work.machining ci=sparsity;
    LUOKKA Machine Shift;
    NIMIKE Deviation = "Poikkeama (mikronia)"
          Machine = "Kone"
          Shift = "Vuoro"
          ToolWear = "Työkalun kuluminen (min)"
          CycleSpeed = "Kiertonopeus (rpm)"
          CoolantTemp = "Jäähdytysnesteen lämpötila (aste C)"
          Humidity = "Kosteus (%RH)"
          FeedRate = "Syöttönopeus (mm/kierros)";
    MODEL Deviation = ToolWear CycleSpeed CoolantTemp
                      Humidity FeedRate
        / quantile=0.5 0.90;
SUORITA;

/* Merge the median and tail slopes for each continuous driver */
TIEDOT work.COMPARE;
    SÄILYTÄ Parameter MedianSlope TailSlope TailMinusMedian;
    YHDISTÄ
        work.pe(MISSÄ=(Quantile=0.5)
                SÄILYTÄ=Quantile Parameter ESTIMATE
                NIMEÄ_UUDELLEEN=(ESTIMATE=MedianSlope))
        work.pe(MISSÄ=(Quantile=0.9)
                SÄILYTÄ=Quantile Parameter ESTIMATE
                NIMEÄ_UUDELLEEN=(ESTIMATE=TailSlope));
    MUKAAN Parameter;
    TailMinusMedian = TailSlope - MedianSlope;
SUORITA;

PROSEDUURI TULOSTA TIEDOT=work.COMPARE noobs NIMIKE;
    NIMIKE Parameter       = "Tekijä"
          MedianSlope     = "Kulmakerroin @ q=0,50"
          TailSlope       = "Kulmakerroin @ q=0,90"
          TailMinusMedian = "Häntä - Mediaani";
    MUOTO MedianSlope TailSlope TailMinusMedian 10.4;
SUORITA;


The QUANTREG Procedure

Quantile: 0.5000
CI Method: SPARSITY
Dependent Variable: Poikkeama (mikronia)

Parameter           Estimate       StdErr        Lower        Upper
Intercept            -4.4131       2.7370      -9.7776       0.9514
Työkalun kuluminen (min)       0.0146       0.0045       0.0057       0.0235
Kiertonopeus (rpm)      -0.0000       0.0011      -0.0021       0.0021
Jäähdytysnesteen lämpötila (aste C)       0.4838       0.0528       0.3802       0.5873
Kosteus (%RH)         0.0678       0.0203       0.0280       0.1076
Syöttönopeus (mm/kierros)      -6.3520       7.7910     -21.6221       8.9181
Intercept            -5.3307       1.2671      -7.8142      -2.8471
Työkalun kuluminen (min)       0.0416       0.0021       0.0375       0.0457
Kiertonopeus (rpm)       0.0008       0.0005      -0.0002       0.0018
Jäähdytysnesteen lämpötila (aste C)       0.3907       0.0245       0.3428       0.4387
Kosteus (%RH)         0.0807       0.0094       0.0623       0.0991
Syöttö


NOTE: ODS OUTPUT: PARAMETERESTIMATES -> pe
NOTE: PROC QUANTREG data=work.machining

NOTE: PROC QUANTREG completed.
NOTE: DATA work.compare

NOTE: Stream 1 processed 6 rows, max BY-group size: 1 (O(1) memory verified)
NOTE: Stream 2 processed 6 rows, max BY-group size: 1 (O(1) memory verified)

NOTE: Wrote work.compare (6 rows, 4 columns).
NOTE: DATA elapsed:
  wall  0.08 seconds
  cpu   0.08 seconds
NOTE: PROC PRINT data=work.compare

NOTE: PROC PRINT completed: 6 observations printed, 4 variables


## Vaihe 4 — Jokaisen kappaleen pisteytys 90. persentiilissä

`OUTPUT`-lause kirjoittaa sovitetun 90. persentiilin ennusteen takaisin ulos, yksi rivi kappaletta kohden, ennusteen keskivirheen (`STDP`) ja tarkistushäviön jäännöksen kanssa. Kuljetamme muuttujan `PartID` mukana `ID`-lauseella ja kopioimme kaksi hallitsevaa tekijää, jotta analyytikot voivat lajitella riskialttiimmat kappaleet ylimmäksi. Pieni PROC PRINT näyttää ensimmäiset pisteytetyt kappaleet.

In [5]:
PROSEDUURI quantreg TIEDOT=work.machining ci=sparsity;
    LUOKKA Machine Shift;
    id PartID;
    NIMIKE Deviation = "Poikkeama (mikronia)"
          Machine = "Kone"
          Shift = "Vuoro"
          ToolWear = "Työkalun kuluminen (min)"
          CycleSpeed = "Kiertonopeus (rpm)"
          CoolantTemp = "Jäähdytysnesteen lämpötila (aste C)"
          Humidity = "Kosteus (%RH)"
          FeedRate = "Syöttönopeus (mm/kierros)";
    MODEL Deviation = Machine Shift ToolWear CycleSpeed CoolantTemp
                      FeedRate
        / quantile=0.90;
    TULOSTE out=work.scored
        predicted=PredP90
        stdp=SEPred
        residual=Resid;
SUORITA;

PROSEDUURI TULOSTA TIEDOT=work.scored(obs=10) noobs NIMIKE;
    MUUTTUJA PartID Machine ToolWear CycleSpeed PredP90 SEPred Resid;
    NIMIKE PredP90 = "Ennustettu 90. persentiilin poikkeama"
          SEPred  = "Ennusteen keskivirhe"
          Resid   = "Tarkistushäviön jäännös";
SUORITA;


The QUANTREG Procedure

Quantile: 0.9000
CI Method: SPARSITY
Dependent Variable: Poikkeama (mikronia)

Parameter           Estimate       StdErr        Lower        Upper
Intercept            -3.9687       0.6322      -5.2078      -2.7295
MACHINE M3            0.6729       0.1246       0.4287       0.9171
MACHINE M2           -0.4499       0.1117      -0.6689      -0.2310
MACHINE M1           -1.1957       0.1109      -1.4131      -0.9784
SHIFT EVE            -3.1741       0.1034      -3.3768      -2.9713
SHIFT DAY            -1.8083       0.1017      -2.0075      -1.6090
Työkalun kuluminen (min)       0.0438       0.0012       0.0416       0.0461
Kiertonopeus (rpm)       0.0037       0.0003       0.0032       0.0043
Jäähdytysnesteen lämpötila (aste C)       0.3377       0.0133       0.3116       0.3638
Syöttönopeus (mm/kierros)      14.9464       2.0482      10.9321      18.9608


PARTID        TOOLWEAR       CYCLESPEED  Ennustettu 90. persentiilin poikkeama  Ennusteen keskivirhe    


NOTE: PROC QUANTREG data=work.machining

NOTE: PROC QUANTREG completed.
NOTE: PROC PRINT data=work.scored

NOTE: PROC PRINT completed: 10 observations printed, 6 variables


## Tulosten tulkinta

**Häntä kertoo eri tarinan kuin keskikohta.** Vertaamalla kahta kertoimiryhmää (vaihe 2) ja yhdistettyä taulukkoa (vaihe 3), muuttujien `ToolWear`, `CycleSpeed` ja `FeedRate` kulmakertoimet ovat selvästi suurempia 90. persentiilissä kuin mediaanissa. Se on data-generointimekanismi näkyväksi tehtynä: koska rakensimme virhehajonnan kasvamaan kulumisen ja nopeuden myötä, nämä tekijät tuskin siirtävät mediaanipoikkeamaa mutta vahvistavat voimakkaasti hylkyalttiin yläkvantiilin. Keskiarvopohjainen regressio olisi painottanut liian vähän juuri niitä vipuja, jotka merkitsevät hylyn kannalta.

**`ToolWear`-signaali on selkein.** Sen kulmakerroin lähes kolminkertaistuu mediaanisovituksesta (0,015) häntäsovitukseen (0,042), ja 90. persentiilin luottamuskaista on selvästi nollan yläpuolella — kuluminen on yksittäisenä tekijänä luotettavin korkean hännän riskin ennustaja. `CycleSpeed`, joka on mediaanissa lähes tasainen, muuttuu positiiviseksi hännässä, mikä sopii sen rooliin hajontatermissä.

**Kvantiiliregressio erottaa sijainnin hajonnasta.** `CoolantTemp` esiintyy sijaintitermissä mutta ei hajontatermissä, joten sen kulmakerroin pysyy lähellä 0,4–0,5 mikronia astetta kohden molemmissa kvantiileissa — se siirtää koko jakaumaa sen sijaan, että levittäisi häntää. Muuttujalla `Humidity` ei ole todellista signaalia data-generointiprosessissa, mutta vain 100 kappaleella se saa pienen näennäisen kulmakertoimen; sen `Tail - Median` -muutos (0,013) on suuruusluokkaa pienempi kuin `ToolWear`:n (0,027) ja mitätön verrattuna `FeedRate`:n (12,3) muutokseen, joten se ei käyttäydy hännän tekijänä. Rehellinen opetus on, että todellisuudessa nollavaikutteinen muuttuja voi silti näyttää nollasta poikkeavalta pienessä otoksessa — juuri siksi lisensoitu tuotantoajo tuhansien kappaleiden yli tiukentaisi näitä estimaatteja.

**Toiminnallinen hyöty.** `OUTPUT`-ennusteet antavat jokaiselle kappaleelle ennustetun 90. persentiilin poikkeaman keskivirheineen, merkiten korkean hännän riskin kappaleet ennen toimitusta. Toiminnallinen johtopäätös: tiukenna työkalunvaihtovälejä ja rajoita kiertonopeutta tiukkatoleranssisissa töissä, koska molemmat ohjaimet vaikuttavat suoraan mittapoikkeaman hylkyä ohjaavaan yläkvantiiliin.

*Huomio mittakaavasta:* tämä muistikirja toimii lisenssittömällä moottorilla, joka rajoittaa jokaisen vaiheen 100 havaintoon, joten yllä olevat kulmakertoimet on estimoitu 100 kappaleesta ja häntäsovitus on väistämättä kohinaisempi kuin täyden tuotannon ajossa. Keskikohdan ja hännän välinen ero on jo näkyvissä tässä koossa; lisensoitu ajo koko kappalevirran yli tiukentaisi jokaista luottamuskaistaa.